In [ ]:
import os
from openai import OpenAI
from dotenv import load_dotenv

In [ ]:
def ask_chatgpt(text: str, prompt_path=None):
    # Check if a prompt path is provided and read prompt text
    _ = load_dotenv()
    api_key =os.getenv("OPENAI_API_KEY")
    client = OpenAI(api_key = api_key)
    if prompt_path and text:
        with open(prompt_path, 'r') as file:
            prompt = file.read().strip()
    else:
        return "Error: no data given"

    # Model configuration - replace 'gpt-4' with the specific model if needed
    model_name = "gpt-4o-2024-11-20"  # or "gpt-3.5-turbo" if you want a different model
    # Send the prompt to the model
    response = client.chat.completions.create(
    model=model_name,
    messages=[
        {'role' : "system" , "content" : prompt },
        {'role': "user" , "content": text}
    ],
    temperature=0.1,
    top_p=0.2
    )
        
    # Return the response content
    return response.choices[0].message.content

In [ ]:
score_tab = """
table1:
| Distance (yards) | Greyhound | Time | Date |\n| ---------------- | --------- | ---- | ---- |\n| 325 | Lemon Clover | 17.34 | 11.10.1996 |\n| 525 | Whitty Guinness | 28.54 | 29.10.2010 |\n| 550 | Whatsupjack | 29.91 | 18.09.2009 |\n| 700 | Tinas Girl | 38.79 | 19.08.2003 |\n| 790 | Shining Rumble | 44.76 | 13.07.2004 |\n
table2:
\n| Distance (meters) | Greyhound     | Time  | Date       |\n| ----------------- | ------------- | ----- | ---------- |\n| 297.48            | Lemon Clover  | 17.34 | 11.10.1996 |\n| 480.21            | Whitty Guinness | 28.54 | 29.10.2010 |\n| 502.92            | Whatsupjack   | 29.91 | 18.09.2009 |\n| 640.08            | Tinas Girl    | 38.79 | 19.08.2003 |\n| 722.62            | Shining Rumble| 44.76 | 13.07.2004 |\n
"""

In [ ]:
pscore = ask_chatgpt(score_tab,'p-score.txt')

In [ ]:
import json
def TableAlign(table1, table2):
    return f"Gold Table:\n\"\"\"{table1}\"\"\"\n\n Predicted Table:\n\"\"\"{table2}\"\"\"\n"


with open('tables.json', 'r') as file:
    dataset = json.load(file)

for data in dataset:
    original = data.get("original", "")
    for i in range(5):  # Assuming there are up to 5 perturbations (0-4)
        perturbation_key = f"pertubation{i}"
        result_key = f"result{i}"
        if perturbation_key in data:
            result = ask_chatgpt(TableAlign(original, data[perturbation_key]), 'p-score.txt')  
            data[result_key] = result
            print(data[f"result{i}"])

# Write the updated dataset to a new JSON file
with open('p_score_syn.json', 'w') as file:
    json.dump(dataset, file, indent=2)

print("Updated JSON file 'p_score_syn.json' has been created successfully.")

# # Print results
# for result in results:
#     print(result)
#     print("-" * 50)

In [ ]:
import re
import json
def format_result(pscore):
    json_match = re.search(r'```json\n(\{.*?\})\n```', pscore, re.DOTALL)

    if json_match:
        extracted_json = json_match.group(1)
        parsed_json = json.loads(extracted_json)
        print(parsed_json)
        return parsed_json
    else:
        return ""

In [ ]:
with open('p_score_syn.json', 'r') as file:
    dataset = json.load(file)

for data in dataset:
    for i in range(5):  # Assuming there are up to 5 perturbations (0-4)
        perturbation_key = f"pertubation{i}"
        result_key = f"result{i}"
        if result_key in data:
            result = format_result(data[result_key])
            data[result_key] = result
            print(data[f"result{i}"])

# Write the updated dataset to a new JSON file
with open('p_score_syn_format.json', 'w') as file:
    json.dump(dataset, file, indent=2)
